# PB1 - ProofBind verification: is the family-transfer conclusion bound AND significant?
**Question.** Apply the ProofBind discipline to our own headline: (1) is MCC 0.317 correctly computed (binding), and (2) is the protein-family-transfer CLAIM statistically established? Re-derives everything from the COMMITTED raw artifacts, independently of the scripts that produced them.

*Data: per-fold `mmpartnet_out/famfull3/*/{val,test}_metrics.csv` + `test_preds.csv`; `famfull3_depeek.json`, `famfull_confounds.json`, `famfull3_nearvsfar.json`.*

## Definitions / math
- **MCC** $=\frac{TP\,TN-FP\,FN}{\sqrt{(TP+FP)(TP+FN)(TN+FP)(TN+FN)}}$.
- **Binding (B1-B4):** B1 the reported best epoch = independent $\arg\max_e$ VAL-F1; B2 test-metrics MCC at that epoch = reported; B3 a SECOND path (predict.py per-pair preds) agrees; B4 label-shuffle counterexample MUST fire (MCC$\to 0$).
- **Honest de-peek** = mean over folds of test MCC @ val-selected epoch; fold-bootstrap 95% CI.
- **Paired residual** = per-fold (honest - fold-matched RNA-only); sign test + bootstrap CI.
- **Near-vs-far** = family-block permutation p for Pearson(per-family MCC, nearest-train %id).

In [ ]:
import json, csv, math, random
from pathlib import Path
import numpy as np
from IPython.display import Markdown, display
OUT = Path('..')/'..'/'mmpartnet_out'; FF = OUT/'famfull3'
def mcc(pred,y):
    pred=np.asarray(pred); y=np.asarray(y)
    tp=int(((pred==1)&(y==1)).sum()); tn=int(((pred==0)&(y==0)).sum()); fp=int(((pred==1)&(y==0)).sum()); fn=int(((pred==0)&(y==1)).sum())
    d=math.sqrt((tp+fp)*(tp+fn)*(tn+fp)*(tn+fn)); return (tp*tn-fp*fn)/d if d else 0.0
dep={f['fold']:f for f in json.loads((OUT/'famfull3_depeek.json').read_text())['per_fold']}
tab='| fold | argmax VAL-ep | reported best-ep | MCC@val (metrics) | MCC (preds path) | label-shuffle |\n|---|---|---|---|---|---|\n'
allok=True
for K in range(5):
    vm=list(csv.DictReader(open(FF/str(K)/'val_metrics.csv'))); tm=list(csv.DictReader(open(FF/str(K)/'test_metrics.csv'))); pr=list(csv.DictReader(open(FF/str(K)/'test_preds.csv')))
    amax=max(((int(r['Epoch']),float(r['F1'])) for r in vm),key=lambda x:x[1])[0]
    mccv={int(r['Epoch']):float(r['MCC']) for r in tm}[amax]
    y=[int(float(r['true_label'])) for r in pr]; pp=[int(float(r['prediction'])) for r in pr]
    mp=mcc(pp,y); ys=y[:]; random.Random(0).shuffle(ys); msh=mcc(pp,ys)
    b=(amax==dep[K]['best_val_epoch']) and abs(mccv-dep[K]['test_MCC_at_val'])<1e-6 and abs(mp-dep[K]['test_MCC_at_val'])<0.02 and abs(msh)<0.05
    allok=allok and b
    tab+=f"| {K} | {amax} | {dep[K]['best_val_epoch']} | {mccv:.3f} | {mp:.3f} | {msh:+.3f} |\n"
display(Markdown(tab)); display(Markdown(f'**Binding B1-B4: {"ALL PASS" if allok else "FAIL"}** (val-selection genuine; two code paths agree; label-shuffle collapses -> metric real).'))

In [ ]:
# significance: de-peek CI + paired residual vs fold-matched RNA-only
conf={f['fold']:f['rna_only_baseline_bestMCC'] for f in json.loads((OUT/'famfull_confounds.json').read_text())['per_fold']}
hon=np.array([dep[k]['test_MCC_at_val'] for k in range(5)]); rna=np.array([conf[k] for k in range(5)]); diff=hon-rna
rng=np.random.default_rng(0)
bh=[np.mean(rng.choice(hon,5,True)) for _ in range(10000)]; bd=[np.mean(rng.choice(diff,5,True)) for _ in range(10000)]
hlo,hhi=np.percentile(bh,[2.5,97.5]); dlo,dhi=np.percentile(bd,[2.5,97.5]); npos=int((diff>0).sum())
display(Markdown(f'**Honest de-peek MCC = {hon.mean():.3f}, 95% CI [{hlo:.3f},{hhi:.3f}].** '
  f'Paired residual over fold-matched RNA-only = {diff.mean():+.3f}, 95% CI [{dlo:+.3f},{dhi:+.3f}] '
  f'({npos}/5 folds positive) -> **{"EXCLUDES 0" if dlo>0 else "INCLUDES 0 (NOT significant)"}**.'))
nf=json.loads((OUT/'famfull3_nearvsfar.json').read_text())['per_family']
x=np.array([r['nearest_id'] for r in nf]); yv=np.array([r['mcc'] for r in nf]); obs=np.corrcoef(x,yv)[0,1]
null=np.array([np.corrcoef(x,rng.permutation(yv))[0,1] for _ in range(10000)]); pp=(np.sum(np.abs(null)>=abs(obs))+1)/(len(null)+1)
display(Markdown(f'**Near-vs-far:** Pearson(MCC, nearest-train %id) = {obs:.3f}, family-block permutation p = {pp:.3f} -> {"distance-dependent" if pp<0.05 else "NO distance dependence (interpolation refuted)"}.'))

## Conclusion -- ProofBind verdict
**MECHANICALLY BOUND, INTERPRETIVELY NOT-ESTABLISHED.** The number is correct (B1-B4 pass; label-shuffle fires). But the protein-family-transfer CLAIM fails: the honest de-peek (0.317) is not separable from a protein-blind RNA-only baseline (paired residual +0.08, CI includes 0, 2/5 folds), and the effect is distance-independent (permutation p~0.29). The protein-derangement null (F2) shows the model *does* condition on the protein, so the precise statement is **used-but-non-generalizing**: protein identity as a lookup key, not protein-family structure.

**Tightest honest claim:** on a protein-family-disjoint, de-peeked split the CORAL interaction model scores MCC 0.317 [0.196,0.443], not distinguishable from RNA-bindability memorization; no protein-family-transfer effect is established (n=5 -> absence of evidence). Decisive open tests: RNA+family doubly-disjoint split; M2 profile-shape on a leave-out-pretrained PARNET body.